In [ ]:
# IndicTTS Deepfake Challenge Dataset Model Training

import os
import random
import shutil
import torch
import numpy as np
from scipy.signal import butter, lfilter
from dotenv import load_dotenv
from datasets import load_dataset
from transformers import (
    Wav2Vec2ForSequenceClassification,
    Wav2Vec2FeatureExtractor,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

# 1. DIRECTORY & CACHE CONFIGURATION (Safe E-Drive Mapping)
BASE_DIR = r"E:\SIH2026\models"
CACHE_DIR = os.path.join(BASE_DIR, "temp_cache")
TMP_DIR = os.path.join(BASE_DIR, "temp_files")

os.environ["HF_HOME"] = CACHE_DIR
os.environ["HF_DATASETS_CACHE"] = os.path.join(CACHE_DIR, "datasets")
os.environ["HUGGINGFACE_HUB_CACHE"] = os.path.join(CACHE_DIR, "hub")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TEMP"] = TMP_DIR
os.environ["TMP"] = TMP_DIR

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
MODEL_NAME = "facebook/wav2vec2-large-xlsr-53"
TARGET_LENGTH = 48000  # 3 seconds @ 16kHz

# Sequential Pipeline Execution Mapping
PIPELINES = [
    {"source_folder": "hindi", "save_folder": "trained_hindi"},
    {"source_folder": "bengali", "save_folder": "trained_bengali"},
    {"source_folder": "telugu", "save_folder": "trained_telegu"} # can add more 
]

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME, token=HF_TOKEN)

# 2. ADVERSARIAL AUDIO PREPROCESSING UTILITIES
def apply_lowpass_filter(data, cutoff_freq, sampling_rate=16000, order=5):
    nyquist = 0.5 * sampling_rate
    normal_cutoff = cutoff_freq / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return lfilter(b, a, data).astype(np.float32)

def build_preprocessors(fake_idx):
    def preprocess_train(examples):
        audio_arrays, labels = [], []
        for audio_item, orig_label in zip(examples["audio"], examples["label"]):
            raw = np.array(audio_item["array"], dtype=np.float32)
            # Label standardization: 0 = Real, 1 = Fake
            labels.append(1 if orig_label == fake_idx else 0)

            # Strict 3-Second Window
            if len(raw) > TARGET_LENGTH:
                start = random.randint(0, len(raw) - TARGET_LENGTH)
                raw = raw[start : start + TARGET_LENGTH]
            elif len(raw) < TARGET_LENGTH:
                pad = np.random.normal(0, 0.001, TARGET_LENGTH - len(raw)).astype(np.float32)
                raw = np.concatenate((raw, pad))

            # Noise Augmentation
            if random.random() > 0.5:
                raw = (raw + np.random.normal(0, 0.002, raw.shape).astype(np.float32)) * random.uniform(0.7, 1.1)

            # Frequency Low-Pass Masking
            if random.random() > 0.5:
                raw = apply_lowpass_filter(raw, cutoff_freq=random.uniform(4000, 6000))

            audio_arrays.append(raw)

        inputs = feature_extractor(
            audio_arrays,
            sampling_rate=16000,
            padding=False,
            truncation=False,
            return_attention_mask=False
        )
        inputs["labels"] = labels
        return inputs

    def preprocess_val(examples):
        audio_arrays, labels = [], []
        for audio_item, orig_label in zip(examples["audio"], examples["label"]):
            raw = np.array(audio_item["array"], dtype=np.float32)
            labels.append(1 if orig_label == fake_idx else 0)

            # Center Crop / Pad
            if len(raw) > TARGET_LENGTH:
                start = (len(raw) - TARGET_LENGTH) // 2
                raw = raw[start : start + TARGET_LENGTH]
            elif len(raw) < TARGET_LENGTH:
                raw = np.concatenate((raw, np.zeros(TARGET_LENGTH - len(raw), dtype=np.float32)))

            audio_arrays.append(raw)

        inputs = feature_extractor(
            audio_arrays,
            sampling_rate=16000,
            padding=False,
            truncation=False,
            return_attention_mask=False
        )
        inputs["labels"] = labels
        return inputs

    return preprocess_train, preprocess_val

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = (preds == labels).astype(np.float32).mean().item()
    return {"accuracy": acc}

# 3. EXECUTE TRAINING LOOP
for pipe in PIPELINES:
    data_path = os.path.join(BASE_DIR, pipe["source_folder"])
    model_save_path = os.path.join(BASE_DIR, pipe["save_folder"])

    if not os.path.exists(data_path):
        print(f"\n[SKIP] Folder not found: {data_path}")
        continue

    print("\n" + "="*75)
    print(f"STARTING TRAINING: {pipe['source_folder'].upper()} -> {model_save_path}")
    print("="*75)

    # 1. Load Partitioned Local Data
    raw_datasets = load_dataset("audiofolder", data_dir=data_path)
    train_raw = raw_datasets["train"]
    val_raw = raw_datasets["validation"] if "validation" in raw_datasets else raw_datasets["val"]

    class_names = train_raw.features["label"].names
    fake_idx = class_names.index("fake")
    print(f"Classes: {class_names} | Auto-Fake Index: {fake_idx}")

    # 2. Map Dataset
    prep_train, prep_val = build_preprocessors(fake_idx)
    print("Pre-processing audio features into tensors...")
    train_ds = train_raw.map(prep_train, batched=True, batch_size=8, remove_columns=train_raw.column_names)
    val_ds = val_raw.map(prep_val, batched=True, batch_size=8, remove_columns=val_raw.column_names)

    # 3. Model Initialization & Layer Freezing
    model = Wav2Vec2ForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        token=HF_TOKEN,
        gradient_checkpointing=True,
        use_safetensors=True
    )

    # Freeze base CNN + initial transformer layers, unfreeze top 6 layers
    for param in model.wav2vec2.parameters():
        param.requires_grad = False
    for param in model.wav2vec2.encoder.layers[-6:].parameters():
        param.requires_grad = True

    # 4. Hyperparameters
    training_args = TrainingArguments(
        output_dir=model_save_path,
        eval_strategy="steps",
        eval_steps=100,
        save_steps=100,
        learning_rate=3e-5,
        max_grad_norm=1.0,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        warmup_steps=100,
        weight_decay=0.01,
        logging_steps=25,
        fp16=torch.cuda.is_available(),
        report_to="none",
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        save_total_limit=1
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    # 5. Train & Export
    trainer.train()
    print(f"\nExporting finalized model checkpoint to {model_save_path}...")
    trainer.save_model(model_save_path)
    feature_extractor.save_pretrained(model_save_path)

    # Clean VRAM after each language to prevent Out-Of-Memory (OOM)
    del model, trainer, train_ds, val_ds
    torch.cuda.empty_cache()

# Cleanup temporary disk cache
if os.path.exists(CACHE_DIR):
    shutil.rmtree(CACHE_DIR, ignore_errors=True)
if os.path.exists(TMP_DIR):
    shutil.rmtree(TMP_DIR, ignore_errors=True)

print("\nAll Indic language models trained and exported successfully.")

In [ ]:
# Code for Model training on IndicSynth dataset
import os
import torch
import numpy as np
import shutil
import random
from scipy.signal import butter, lfilter
from dotenv import load_dotenv
from datasets import load_from_disk
from transformers import (
    Wav2Vec2ForSequenceClassification,
    Wav2Vec2FeatureExtractor,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

# 1. ENVIRONMENT & CONFIGURATION
BASE_DIR = r"E:\SIH2026\models"
CACHE_DIR = os.path.join(BASE_DIR, "temp_cache")

os.environ["HF_HOME"] = CACHE_DIR
os.environ["HF_HUB_CACHE"] = CACHE_DIR
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

LANGUAGE = "bengali"  
DATA_DIR = os.path.join(BASE_DIR, LANGUAGE) 
OUTPUT_DIR = os.path.join(BASE_DIR, f"trained_{LANGUAGE}") 

MODEL_NAME = "facebook/wav2vec2-large-xlsr-53"
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

print(f"Loading local dataset from {DATA_DIR}...")
train_ds = load_from_disk(os.path.join(DATA_DIR, "train"))
val_ds = load_from_disk(os.path.join(DATA_DIR, "validation"))

# 2. SIGNAL PROCESSING HELPER FUNCTIONS
def apply_lowpass_filter(data, cutoff_freq, sampling_rate=16000, order=5):
    """
    Butterworth low-pass filter to remove high-frequency vocoder artifacts.
    """
    nyquist = 0.5 * sampling_rate
    normal_cutoff = cutoff_freq / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    filtered_data = lfilter(b, a, data)
    return filtered_data.astype(np.float32)

# 3. FEATURE EXTRACTION & PREPROCESSING
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME, token=HF_TOKEN)

TARGET_LENGTH = 48000  # Exactly 3 seconds at 16kHz

# FOR TRAINING: Strict Cropping + Noise + Frequency Masking
def preprocess_train(examples):
    audio_arrays = []
    for x in examples["audio"]:
        raw_audio = x["array"]
        
        # A. Strict Fixed-Length Enforcement
        if len(raw_audio) > TARGET_LENGTH:
            start_idx = random.randint(0, len(raw_audio) - TARGET_LENGTH)
            raw_audio = raw_audio[start_idx : start_idx + TARGET_LENGTH]
        elif len(raw_audio) < TARGET_LENGTH:
            padding_length = TARGET_LENGTH - len(raw_audio)
            noise_pad = np.random.normal(0, 0.001, padding_length)
            raw_audio = np.concatenate((raw_audio, noise_pad))

        # B. Amplitude/Noise Augmentation (Destroys clean background shortcut)
        if random.random() > 0.5:
            noise = np.random.normal(0, 0.002, raw_audio.shape)
            raw_audio = raw_audio + noise
            volume_factor = random.uniform(0.7, 1.1)
            raw_audio = raw_audio * volume_factor
            
        # C. Frequency Masking (Destroys high-freq vocoder artifact shortcut)
        if random.random() > 0.5:
            random_cutoff = random.uniform(4000, 6000) # Cut off anything above 4-6kHz
            raw_audio = apply_lowpass_filter(raw_audio, cutoff_freq=random_cutoff, sampling_rate=16000)

        audio_arrays.append(raw_audio)

    # D. Completely disable padding and attention masks
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=16000,
        padding=False,
        truncation=False,
        return_attention_mask=False  
    )
    inputs["labels"] = examples["label"]
    return inputs

# FOR VALIDATION: Strict Cropping ONLY (Evaluates on clean data)
def preprocess_val(examples):
    audio_arrays = []
    for x in examples["audio"]:
        raw_audio = x["array"]
        
        # A. Strict Center Cropping
        if len(raw_audio) > TARGET_LENGTH:
            start_idx = (len(raw_audio) - TARGET_LENGTH) // 2
            raw_audio = raw_audio[start_idx : start_idx + TARGET_LENGTH]
        elif len(raw_audio) < TARGET_LENGTH:
            padding_length = TARGET_LENGTH - len(raw_audio)
            zero_pad = np.zeros(padding_length)
            raw_audio = np.concatenate((raw_audio, zero_pad))
            
        audio_arrays.append(raw_audio)

    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=16000,
        padding=False,
        truncation=False,
        return_attention_mask=False  
    )
    inputs["labels"] = examples["label"]
    return inputs

print("Applying Adversarial Signal Processing (Cropping, Noise, & Filtering)...")
train_ds = train_ds.map(preprocess_train, batched=True, batch_size=8, remove_columns=train_ds.column_names)
val_ds = val_ds.map(preprocess_val, batched=True, batch_size=8, remove_columns=val_ds.column_names)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    acc = (preds == labels).astype(np.float32).mean().item()
    return {"accuracy": acc}

# 4. MODEL SETUP & TRAINING
print("Loading model and configuring layers...")
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    token=HF_TOKEN,
    gradient_checkpointing=True,
    use_safetensors=True
)

# Freeze base, unfreeze top 6 encoder layers for feature extraction
for param in model.wav2vec2.parameters():
    param.requires_grad = False

for param in model.wav2vec2.encoder.layers[-6:].parameters():
    param.requires_grad = True

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="steps",  
    eval_steps=100,             
    save_steps=100,
    learning_rate=3e-5,          
    max_grad_norm=1.0,           
    per_device_train_batch_size=4, 
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=5,          
    warmup_steps=100,            
    weight_decay=0.01,          
    logging_steps=25,
    fp16=torch.cuda.is_available(), 
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f"\nStarting robust training on {LANGUAGE.upper()}...")
trainer.train()

print(f"Saving stabilized model to {OUTPUT_DIR}...")
trainer.save_model(OUTPUT_DIR)
feature_extractor.save_pretrained(OUTPUT_DIR)

if os.path.exists(CACHE_DIR):
    shutil.rmtree(CACHE_DIR)

print("Done! Model is now forced to analyze core prosody and formants.")

In [ ]:
# Combined code for model training on IndicSynth and IndicTTS-Deepfake-Challenge Dataset. (for Indian Languages)
# 0. KERNEL CLEANER & PERFECT LZMA MOCK
import sys
import types
for k in list(sys.modules.keys()):
    if k.startswith("datasets") or k.startswith("fsspec") or k.startswith("sklearn") or k == "lzma" or k == "_lzma":
        del sys.modules[k]

fake_lzma = types.ModuleType("lzma")
fake_lzma.LZMAError = Exception
class FakeLZMAFile:
    def __init__(self, *args, **kwargs): pass
    def read(self, *args, **kwargs): return b""
    def write(self, *args, **kwargs): return 0
    def seek(self, *args, **kwargs): return 0
    def tell(self, *args, **kwargs): return 0
    def close(self): pass
    def __enter__(self): return self
    def __exit__(self, *args): pass
fake_lzma.LZMAFile = FakeLZMAFile
fake_lzma.open = lambda *args, **kwargs: FakeLZMAFile()
fake_lzma.is_check_supported = lambda check: False
fake_lzma.FORMAT_AUTO, fake_lzma.FORMAT_XZ, fake_lzma.FORMAT_ALONE, fake_lzma.FORMAT_RAW = 0, 1, 2, 3
fake_lzma.CHECK_NONE, fake_lzma.CHECK_CRC32, fake_lzma.CHECK_CRC64, fake_lzma.CHECK_SHA256 = 0, 1, 4, 10
sys.modules["lzma"] = fake_lzma
sys.modules["_lzma"] = fake_lzma
sys.modules["torchvision"] = None

# 1. COMBINED MULTILINGUAL TRAINING LOOP
import os
import random
import shutil
import torch
import numpy as np
from scipy.signal import butter, lfilter
from dotenv import load_dotenv
from datasets import load_dataset, load_from_disk, concatenate_datasets, Value, Audio
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor, TrainingArguments, Trainer, EarlyStoppingCallback

BASE_DIR = r"E:\SIH2026\models"
CACHE_DIR = os.path.join(BASE_DIR, "temp_cache")
TMP_DIR = os.path.join(BASE_DIR, "temp_files")
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

os.environ["HF_HOME"] = CACHE_DIR
os.environ["HF_DATASETS_CACHE"] = os.path.join(CACHE_DIR, "datasets")
os.environ["HUGGINGFACE_HUB_CACHE"] = os.path.join(CACHE_DIR, "hub")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
MODEL_NAME = "facebook/wav2vec2-large-xlsr-53"
TARGET_LENGTH = 48000

PIPELINES = [
    {"lang": "HINDI", "tts_folder": "hindi_indicTTS", "synth_folder": "hindi_indicsynth", "save_folder": "trained_combined_hindi"},
    {"lang": "BENGALI", "tts_folder": "bengali_indicTTS", "synth_folder": "bengali_indicsynth", "save_folder": "trained_combined_bengali"},
    {"lang": "TELUGU", "tts_folder": "telugu_indicTTS", "synth_folder": "telugu_indicsynth", "save_folder": "trained_combined_telugu"}
]

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME, token=HF_TOKEN)

def standardize_schema(ds):
    fake_val = 1
    if "label" in ds.features and hasattr(ds.features["label"], "names"):
        names = [n.lower() for n in ds.features["label"].names]
        if "fake" in names: fake_val = names.index("fake")
        elif "spoof" in names: fake_val = names.index("spoof")

    def map_row(example):
        if "label" in example: return {"unified_label": 1 if example["label"] == fake_val else 0}
        elif "is_tts" in example: return {"unified_label": int(example["is_tts"])}
        return {"unified_label": 0}

    mapped_ds = ds.map(map_row, desc="Standardizing Labels")
    cols_to_keep = ["audio", "unified_label"]
    cols_to_remove = [c for c in mapped_ds.column_names if c not in cols_to_keep]
    mapped_ds = mapped_ds.remove_columns(cols_to_remove).rename_column("unified_label", "label")
    mapped_ds = mapped_ds.cast_column("label", Value("int64")).cast_column("audio", Audio(sampling_rate=16000, mono=True, decode=True))
    return mapped_ds

def apply_lowpass_filter(data, cutoff_freq, sampling_rate=16000, order=5):
    nyquist = 0.5 * sampling_rate
    normal_cutoff = cutoff_freq / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return lfilter(b, a, data).astype(np.float32)

def build_preprocessors():
    def preprocess_train(examples):
        audio_arrays, labels = [], []
        for audio_item, orig_label in zip(examples["audio"], examples["label"]):
            raw = np.array(audio_item["array"], dtype=np.float32)
            labels.append(orig_label)
            if len(raw) > TARGET_LENGTH:
                start = random.randint(0, len(raw) - TARGET_LENGTH)
                raw = raw[start : start + TARGET_LENGTH]
            elif len(raw) < TARGET_LENGTH:
                raw = np.concatenate((raw, np.random.normal(0, 0.001, TARGET_LENGTH - len(raw)).astype(np.float32)))
            if random.random() > 0.5: raw = (raw + np.random.normal(0, 0.002, raw.shape).astype(np.float32)) * random.uniform(0.7, 1.1)
            if random.random() > 0.5: raw = apply_lowpass_filter(raw, cutoff_freq=random.uniform(4000, 6000))
            audio_arrays.append(raw)
        inputs = feature_extractor(audio_arrays, sampling_rate=16000, padding=False, truncation=False, return_attention_mask=False)
        inputs["labels"] = labels
        return inputs

    def preprocess_val(examples):
        audio_arrays, labels = [], []
        for audio_item, orig_label in zip(examples["audio"], examples["label"]):
            raw = np.array(audio_item["array"], dtype=np.float32)
            labels.append(orig_label)
            if len(raw) > TARGET_LENGTH: raw = raw[(len(raw) - TARGET_LENGTH) // 2 : (len(raw) + TARGET_LENGTH) // 2]
            elif len(raw) < TARGET_LENGTH: raw = np.concatenate((raw, np.zeros(TARGET_LENGTH - len(raw), dtype=np.float32)))
            audio_arrays.append(raw)
        inputs = feature_extractor(audio_arrays, sampling_rate=16000, padding=False, truncation=False, return_attention_mask=False)
        inputs["labels"] = labels
        return inputs
    return preprocess_train, preprocess_val

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": (preds == labels).astype(np.float32).mean().item()}

for pipe in PIPELINES:
    lang = pipe["lang"]
    tts_path = os.path.join(BASE_DIR, pipe["tts_folder"])
    synth_path = os.path.join(BASE_DIR, pipe["synth_folder"])
    model_save_path = os.path.join(BASE_DIR, pipe["save_folder"])

    if not os.path.exists(tts_path) or not os.path.exists(synth_path):
        print(f"\n[SKIP] Directories missing for {lang}. Ensure both exist.")
        continue

    print("\n" + "="*85)
    print(f"STARTING COMBINED TRAINING: {lang}")
    print("="*85)

    ds_tts = load_dataset("audiofolder", data_dir=tts_path)
    tts_train = standardize_schema(ds_tts["train"])
    tts_val = standardize_schema(ds_tts["validation"] if "validation" in ds_tts else ds_tts.get("test", ds_tts["train"]))

    synth_train = standardize_schema(load_from_disk(os.path.join(synth_path, "train")))
    synth_val = standardize_schema(load_from_disk(os.path.join(synth_path, "validation")))

    combined_train = concatenate_datasets([tts_train, synth_train]).shuffle(seed=42)
    combined_val = concatenate_datasets([tts_val, synth_val]).shuffle(seed=42)

    prep_train, prep_val = build_preprocessors()
    train_ds = combined_train.map(prep_train, batched=True, batch_size=8, remove_columns=combined_train.column_names)
    val_ds = combined_val.map(prep_val, batched=True, batch_size=8, remove_columns=combined_val.column_names)

    model = Wav2Vec2ForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, token=HF_TOKEN, gradient_checkpointing=True, use_safetensors=True)
    for param in model.wav2vec2.parameters(): param.requires_grad = False
    for param in model.wav2vec2.encoder.layers[-6:].parameters(): param.requires_grad = True

    training_args = TrainingArguments(
        output_dir=model_save_path, eval_strategy="steps", eval_steps=150, save_steps=150,
        learning_rate=3e-5, max_grad_norm=1.0, per_device_train_batch_size=4, per_device_eval_batch_size=4,
        gradient_accumulation_steps=4, num_train_epochs=3, warmup_steps=150, weight_decay=0.01,
        logging_steps=50, fp16=torch.cuda.is_available(), report_to="none", load_best_model_at_end=True,
        metric_for_best_model="accuracy", save_total_limit=1
    )

    trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
    trainer.train()
    
    trainer.save_model(model_save_path)
    feature_extractor.save_pretrained(model_save_path)
    del model, trainer, train_ds, val_ds, combined_train, combined_val; torch.cuda.empty_cache()

print("\nSuper-Charged Unified Multilingual Training Completed!")

In [ ]:
# English model training on in-the-wild Dataset
import os
import random
import shutil
import torch
import numpy as np
from scipy.signal import butter, lfilter
from dotenv import load_dotenv
from datasets import load_dataset
from transformers import (
    Wav2Vec2ForSequenceClassification,
    Wav2Vec2FeatureExtractor,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

# 1. PATH CONFIGURATION & CACHE
BASE_DIR = r"E:\SIH2026\models"
DATA_DIR = os.path.join(BASE_DIR, "english")
OUTPUT_DIR = os.path.join(BASE_DIR, "trained_english")
CACHE_DIR = os.path.join(BASE_DIR, "temp_cache")

os.environ["HF_HOME"] = CACHE_DIR
os.environ["HF_HUB_CACHE"] = CACHE_DIR
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

MODEL_NAME = "facebook/wav2vec2-large-xlsr-53"
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

# 2. LOAD DATASET USING AUDIOFOLDER
print(f"Loading partitioned audio dataset from: {DATA_DIR}")
raw_datasets = load_dataset("audiofolder", data_dir=DATA_DIR)

train_raw = raw_datasets["train"]
val_raw = raw_datasets["validation"] if "validation" in raw_datasets else raw_datasets["val"]

# Detect auto-assigned indices
class_names = train_raw.features["label"].names
fake_idx = class_names.index("fake")
real_idx = class_names.index("real")

print(f"Detected Classes: {class_names}")
print(f"Original Index -> fake: {fake_idx}, real: {real_idx}")
print("Mapping to project standard: 0 -> Bonafide (Real), 1 -> Spoof (Fake)")

# 3. SIGNAL PROCESSING & ADVERSARIAL PREPROCESSING
TARGET_LENGTH = 48000  # Exactly 3 seconds @ 16kHz
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME, token=HF_TOKEN)

def apply_lowpass_filter(data, cutoff_freq, sampling_rate=16000, order=5):
    nyquist = 0.5 * sampling_rate
    normal_cutoff = cutoff_freq / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    filtered_data = lfilter(b, a, data)
    return filtered_data.astype(np.float32)

def preprocess_train(examples):
    audio_arrays = []
    labels = []
    
    for audio_item, orig_label in zip(examples["audio"], examples["label"]):
        raw_audio = np.array(audio_item["array"], dtype=np.float32)
        
        # Standardize: real -> 0, fake -> 1
        standardized_label = 1 if orig_label == fake_idx else 0
        labels.append(standardized_label)
        
        # Strict Cropping / Padding
        if len(raw_audio) > TARGET_LENGTH:
            start_idx = random.randint(0, len(raw_audio) - TARGET_LENGTH)
            raw_audio = raw_audio[start_idx : start_idx + TARGET_LENGTH]
        elif len(raw_audio) < TARGET_LENGTH:
            pad_len = TARGET_LENGTH - len(raw_audio)
            noise_pad = np.random.normal(0, 0.001, pad_len).astype(np.float32)
            raw_audio = np.concatenate((raw_audio, noise_pad))
            
        # Noise Augmentation
        if random.random() > 0.5:
            noise = np.random.normal(0, 0.002, raw_audio.shape).astype(np.float32)
            raw_audio = (raw_audio + noise) * random.uniform(0.7, 1.1)
            
        # Frequency Low-Pass Masking
        if random.random() > 0.5:
            cutoff = random.uniform(4000, 6000)
            raw_audio = apply_lowpass_filter(raw_audio, cutoff_freq=cutoff, sampling_rate=16000)
            
        audio_arrays.append(raw_audio)
        
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=16000,
        padding=False,
        truncation=False,
        return_attention_mask=False
    )
    inputs["labels"] = labels
    return inputs

def preprocess_val(examples):
    audio_arrays = []
    labels = []
    
    for audio_item, orig_label in zip(examples["audio"], examples["label"]):
        raw_audio = np.array(audio_item["array"], dtype=np.float32)
        
        standardized_label = 1 if orig_label == fake_idx else 0
        labels.append(standardized_label)
        
        # Center Cropping
        if len(raw_audio) > TARGET_LENGTH:
            start_idx = (len(raw_audio) - TARGET_LENGTH) // 2
            raw_audio = raw_audio[start_idx : start_idx + TARGET_LENGTH]
        elif len(raw_audio) < TARGET_LENGTH:
            pad_len = TARGET_LENGTH - len(raw_audio)
            zero_pad = np.zeros(pad_len, dtype=np.float32)
            raw_audio = np.concatenate((raw_audio, zero_pad))
            
        audio_arrays.append(raw_audio)
        
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=16000,
        padding=False,
        truncation=False,
        return_attention_mask=False
    )
    inputs["labels"] = labels
    return inputs

print("Applying preprocessing to dataset...")
train_dataset = train_raw.map(preprocess_train, batched=True, batch_size=8, remove_columns=train_raw.column_names)
val_dataset = val_raw.map(preprocess_val, batched=True, batch_size=8, remove_columns=val_raw.column_names)

# 4. MODEL INITIALIZATION & LAYER FREEZING
print("Loading base Wav2Vec2-XLSR-53...")
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    token=HF_TOKEN,
    gradient_checkpointing=True,
    use_safetensors=True
)

# Freeze base, train only top 6 transformer layers to make it less GPU heavy
for param in model.wav2vec2.parameters():
    param.requires_grad = False

for param in model.wav2vec2.encoder.layers[-6:].parameters():
    param.requires_grad = True

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = (preds == labels).astype(np.float32).mean().item()
    return {"accuracy": acc}

# 5. TRAINING EXECUTION
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="steps",
    eval_steps=150,
    save_steps=150,
    learning_rate=3e-5,
    max_grad_norm=1.0,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=4,
    warmup_steps=150,
    weight_decay=0.01,
    logging_steps=25,
    fp16=torch.cuda.is_available(),
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("\nStarting Training for English...")
trainer.train()

print(f"\nSaving final model to {OUTPUT_DIR}...")
trainer.save_model(OUTPUT_DIR)
feature_extractor.save_pretrained(OUTPUT_DIR)

if os.path.exists(CACHE_DIR):
    shutil.rmtree(CACHE_DIR)

print("English training finished successfully.")